In [241]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import OrdinalEncoder
from sklearn.naive_bayes import CategoricalNB
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# 1.Multi Classification Naive Bayes 

In [242]:

# =====================================================================
# 1. AMBIL DATA & AMANKAN STRUKTUR
# =====================================================================
# Muat ulang data secara bersih agar tidak bentrok dengan variabel lama
df_baru = pd.read_csv('../0.dataset/dataset_Kinerja_Karyawan_clean.csv')

# Antisipasi jika nama 'Rating_Kinerja' berbeda di file clean Anda
target_col = 'Rating_Kinerja'
if target_col in df_baru.columns:
    X_mentah = df_baru.drop(columns=target_col)
    y_mentah = df_baru[target_col]
else:
    # Jika nama kolom target tidak ketemu, otomatis ambil kolom paling kanan sebagai target
    X_mentah = df_baru.iloc[:, :-1]
    y_mentah = df_baru.iloc[:, -1]

print(f"-> Berhasil memuat data secara bersih!")
print(f"-> Jumlah baris Fitur (X): {X_mentah.shape[0]}, Jumlah baris Target (y): {y_mentah.shape[0]}")

# =====================================================================
# 2. PROSES ENCODING SECARA BERSIH
# =====================================================================
encoder_baru = OrdinalEncoder(categories='auto', dtype=int)
X_angka = encoder_baru.fit_transform(X_mentah)

# =====================================================================
# 3. SPLIT DATA (SINKRONISASI JAMINAN 100%)
# =====================================================================
X_train, X_test, y_train, y_test = train_test_split(X_angka, y_mentah, test_size=0.2, random_state=42)

print(f"-> Setelah Split -> X_train: {X_train.shape[0]} baris, y_train: {y_train.shape[0]} baris\n")

# =====================================================================
# 4. PROSES TUNING HYPERPARAMETER
# =====================================================================
params = {
    'alpha': [0.0, 0.0001, 0.001, 0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0],
    'fit_prior': [True, False]
}

# Inisialisasi model dengan tanda kurung objek yang benar
model_cnb = CategoricalNB()

random_search = RandomizedSearchCV(
    estimator=model_cnb,
    param_distributions=params,
    n_iter=20,
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1
)

# Latih Model menggunakan data yang sudah pasti sinkron jumlah barisnya
random_search.fit(X_train, y_train)
best_model = random_search.best_estimator_

# =====================================================================
# 5. TAMPILKAN HASIL EVALUASI
# =====================================================================
print("="*60)
print(f'Hyperparameter Terbaik : {random_search.best_params_}')
print(f'Akurasi Validasi Terbaik: {random_search.best_score_*100:.2f}%')
print(f'Akurasi Pada Data Train : {best_model.score(X_train, y_train)*100:.2f}%')
print(f'Akurasi Pada Data Test  : {best_model.score(X_test, y_test)*100:.2f}%')
print("="*60)

-> Berhasil memuat data secara bersih!
-> Jumlah baris Fitur (X): 1000, Jumlah baris Target (y): 1000
-> Setelah Split -> X_train: 800 baris, y_train: 800 baris

Hyperparameter Terbaik : {'fit_prior': True, 'alpha': 5.0}
Akurasi Validasi Terbaik: 54.62%
Akurasi Pada Data Train : 55.00%
Akurasi Pada Data Test  : 57.00%
